# Motor Insurance Pricing — Data Cleaning

**Goal of this notebook:** Apply concrete, minimal, documented fixes for the data quality issues identified in the previous notebook (`01_data_exploration.ipynb`). Then, to save a cleaned version of the dataset for later feature engineering / modelling.

**Approach:** Prefer capping over dropping rows wherever reasonable, so as not to unnecessarily discard data. Each fix below is applied only where the earlier EDA gave a clear, evidenced reason to do so.

## 0. Setup

In [2]:
import pandas as pd
import numpy as np

RAW_DATA_DIR = "../data/raw"
PROCESSED_DATA_DIR = "../data/processed"

freq = pd.read_csv(f"{RAW_DATA_DIR}/freMTPL2freq.csv")
sev = pd.read_csv(f"{RAW_DATA_DIR}/freMTPL2sev.csv")

print("freq shape:", freq.shape)
print("sev shape:", sev.shape)

freq shape: (678013, 12)
sev shape: (26639, 2)


## 1. Exposure > 1

1,224 rows had Exposure above 1, deviating from the rest of the data (which peaks cleanly at 1.0). Capping preserves these policies rather than dropping them.

In [2]:
n_over = (freq["Exposure"] > 1).sum()
print(f"Capping {n_over} rows with Exposure > 1")

freq["Exposure"] = freq["Exposure"].clip(upper=1)

Capping 1224 rows with Exposure > 1


## 2. Severity duplicates

255 fully duplicate rows in `sev` were investigated in the EDA notebook and traced to standardized IRSA-IDA flat payout amounts, not logging errors. **Decision: keep as-is**.

## 3. freq / sev ID mismatch

195 severity rows reference a policy ID absent from `freq`. This only matters at the point of joining the two tables as an inner join naturally excludes them. Documenting the decision here rather than modifying either table individually.

In [3]:
unmatched = sev[~sev["IDpol"].isin(freq["IDpol"])]
print(f"{len(unmatched)} severity rows have no matching policy in freq (excluded on join, not dropped here)")

195 severity rows have no matching policy in freq (excluded on join, not dropped here)


## 4. VehAge / DrivAge top-coding

VehAge showed a real gap in the data (values absent between 86-98) with a cluster at 99/100, and DrivAge showed a spike at 99 breaking an otherwise smooth decline. Rather than dropping these rows, flag them as capped/open-ended bands so a model doesn't treat the raw number as literal.

In [4]:
VEHAGE_CAP = 85   # last age before the observed gap
DRIVAGE_CAP = 95  # just below the age-99 spike

freq["VehAge_capped"] = freq["VehAge"].clip(upper=VEHAGE_CAP)
freq["DrivAge_capped"] = freq["DrivAge"].clip(upper=DRIVAGE_CAP)

print("VehAge rows affected:", (freq["VehAge"] > VEHAGE_CAP).sum())
print("DrivAge rows affected:", (freq["DrivAge"] > DRIVAGE_CAP).sum())

VehAge rows affected: 48
DrivAge rows affected: 103


*Note: original VehAge/DrivAge columns are kept alongside the capped versions, so nothing is lost — later notebooks can choose which to use.*

## 5. Save cleaned data

No rows have been dropped in this notebook — only Exposure was adjusted in place, and two new capped columns were added. Saving to a separate `data/processed` folder keeps the original raw files untouched.

In [5]:
import os
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

freq.to_csv(f"{PROCESSED_DATA_DIR}/freMTPL2freq_clean.csv", index=False)
sev.to_csv(f"{PROCESSED_DATA_DIR}/freMTPL2sev_clean.csv", index=False)

print("Saved cleaned datasets to", PROCESSED_DATA_DIR)

Saved cleaned datasets to ../data/processed


## 6. Summary

- Exposure capped at 1 (1,224 rows affected) — no rows dropped.
- Severity duplicates kept as-is (confirmed legitimate flat payouts).
- freq/sev ID mismatch documented; handled at join time, not by row deletion.
- VehAge/DrivAge capped versions added alongside originals for later modeling use.
- No rows were removed from either dataset in this notebook.

In [3]:
# Cleaned Dataset:
print("Final dataset summary")
print(freq.info())
print()
print(sev.info())

Final dataset summary
<class 'pandas.DataFrame'>
RangeIndex: 678013 entries, 0 to 678012
Data columns (total 12 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   IDpol       678013 non-null  float64
 1   ClaimNb     678013 non-null  int64  
 2   Exposure    678013 non-null  float64
 3   VehPower    678013 non-null  int64  
 4   VehAge      678013 non-null  int64  
 5   DrivAge     678013 non-null  int64  
 6   BonusMalus  678013 non-null  int64  
 7   VehBrand    678013 non-null  str    
 8   VehGas      678013 non-null  str    
 9   Area        678013 non-null  str    
 10  Density     678013 non-null  int64  
 11  Region      678013 non-null  str    
dtypes: float64(2), int64(6), str(4)
memory usage: 62.1 MB
None

<class 'pandas.DataFrame'>
RangeIndex: 26639 entries, 0 to 26638
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   IDpol        26639 non-null  int64  
 1